# RAG Retrieval Experiments

This notebook is for iterative retrieval experimentation and optimization.

**Important**: Embeddings are already generated and stored persistently in `data/qdrant_storage/`.
We don't need to regenerate them - just connect to the existing Qdrant database!

## Setup

Connect to the existing Qdrant database with embeddings (no need to regenerate!).

In [7]:
import sys
from pathlib import Path
import json

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / "src"))

from retrieval.embedding import EmbeddingGenerator
from retrieval.qa_index import QdrantIndexer

# Initialize - Connect to PERSISTENT Qdrant (embeddings already stored!)
print("Loading embedding model...")
embedder = EmbeddingGenerator(model_name="all-MiniLM-L6-v2")

print("\nConnecting to persistent Qdrant database...")
indexer = QdrantIndexer(
    collection_name="uk_investment_rag",
    persistence_path=Path("../data/qdrant_storage"),  # Connect to existing storage
    in_memory=False
)

# Check collection info
info = indexer.get_collection_info()
print(f"\nConnected successfully!")
print(f"Collection info: {info}")
print(f"\n✓ Embeddings are already loaded from disk - no need to regenerate!")

2026-04-26 00:17:09,373 - INFO - Loading embedding model: all-MiniLM-L6-v2
2026-04-26 00:17:09,390 - INFO - Use pytorch device_name: mps
2026-04-26 00:17:09,391 - INFO - Load pretrained SentenceTransformer: all-MiniLM-L6-v2


Loading embedding model...


2026-04-26 00:17:11,974 - INFO - Embedding dimension: 384
2026-04-26 00:17:11,975 - INFO - Initializing persistent Qdrant client at ../data/qdrant_storage



Connecting to persistent Qdrant database...


RuntimeError: Storage folder ../data/qdrant_storage is already accessed by another instance of Qdrant client. If you require concurrent access, use Qdrant server instead.

## Experiment 1: Baseline Retrieval

Let's test basic dense retrieval with a few queries.

In [6]:
def search_and_display(query, limit=5):
    """Search for a query and display results."""
    print(f"\n{'='*80}")
    print(f"Query: {query}")
    print(f"{'='*80}")
    
    # Generate embedding
    query_embedding = embedder.generate_embeddings([query])[0]
    
    # Search (in Qdrant, no need to regenerate embeddings)
    results = indexer.search(query_embedding, limit=limit)
    
    # Display results
    for i, result in enumerate(results, 1):
        print(f"\nResult {i} (score: {result['score']:.4f})")
        print(f"  Chunk ID: {result['chunk_id']}")
        print(f"  Company: {result['payload']['company']}")
        print(f"  Product: {result['payload']['product_type']}")
        print(f"  Risk Profile: {result['payload']['risk_profile']}")
        print(f"  Is Table: {result['payload']['is_table']}")
        print(f"  Content: {result['payload']['content'][:200]}...")
    
    return results

In [4]:
# Test queries
queries = [
    "What is the risk profile of Barclays Adventurous Fund?",
    "What are the ongoing charges for Lloyds Balanced Fund?",
    "What is the minimum investment amount for SIPP?",
    "Compare the risk profiles across all Barclays Ready Made Investment funds",
    "Which fund has the best 5-year performance?"
]

for query in queries:
    search_and_display(query, limit=3)

2026-04-25 23:46:20,539 - INFO - Generating embeddings for 1 texts...



Query: What is the risk profile of Barclays Adventurous Fund?


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.05s/it]
2026-04-25 23:46:21,711 - INFO - Generating embeddings for 1 texts...



Result 1 (score: 0.7828)
  Chunk ID: 172c21787e43_0001
  Company: barclays
  Product: ready_made_investment
  Risk Profile: adventurous
  Is Table: False
  Content: Key Investor Information

BARCLAYS

This document provides you with key investor information about this Fund. It is not marketing material. The information is required by law to help you understand th...

Result 2 (score: 0.7672)
  Chunk ID: c31f552b4410_0001
  Company: barclays
  Product: ready_made_investment
  Risk Profile: cautious
  Is Table: False
  Content: Key Investor Information

BARCLAYS

This document provides you with key investor information about this Fund. It is not marketing material. The information is required by law to help you understand th...

Result 3 (score: 0.7378)
  Chunk ID: f570fc38c0d8_0002
  Company: barclays
  Product: ready_made_investment
  Risk Profile: defensive
  Is Table: False
  Content: Key Investor Information

BARCLAYS

This document provides you with key investor information about 

Batches: 100%|██████████| 1/1 [00:00<00:00, 27.62it/s]
2026-04-25 23:46:21,756 - INFO - Generating embeddings for 1 texts...


...


Result 1 (score: 0.6567)
  Chunk ID: 920a2f7450c5_0007
  Company: scottish_widows
  Product: sipp
  Risk Profile: n/a
  Is Table: False
  Content: - An ongoing investment charge of of 0.52% per year and no transaction costs. - No dealing charges have been included. [tbl-8.md](tbl-8.md)
[tbl-9.md](tbl-9.md)

# EXAMPLE 3:
SINGLE CONTRIBUTION PLUS ...

Result 2 (score: 0.6384)
  Chunk ID: fab563a751ae_0003
  Company: scottish_widows
  Product: sipp
  Risk Profile: n/a
  Is Table: False
  Content: - Understand that you are responsible for the investment decisions made within your SIPP, as we operate an execution only (or 'non-advised') service. - A SIPP is aimed at customers who want to make th...

Result 3 (score: 0.6382)
  Chunk ID: fab563a751ae_0006
  Company: scottish_widows
  Product: sipp
  Risk Profile: n/a
  Is Table: False
  Content: - If the value of your SIPP is small and you deal regularly in smaller amounts, dealing costs could be disproportionately high and erode the value o

Batches: 100%|██████████| 1/1 [00:00<00:00,  1.43it/s]
2026-04-25 23:46:22,613 - INFO - Generating embeddings for 1 texts...



Result 1 (score: 0.6587)
  Chunk ID: c31f552b4410_0001
  Company: barclays
  Product: ready_made_investment
  Risk Profile: cautious
  Is Table: False
  Content: Key Investor Information

BARCLAYS

This document provides you with key investor information about this Fund. It is not marketing material. The information is required by law to help you understand th...

Result 2 (score: 0.6435)
  Chunk ID: 2ab6b086c63d_0001
  Company: barclays
  Product: ready_made_investment
  Risk Profile: balanced
  Is Table: False
  Content: Key Investor Information

BARCLAYS

This document provides you with key investor information about this Fund. It is not marketing material. The information is required by law to help you understand th...

Result 3 (score: 0.6432)
  Chunk ID: f570fc38c0d8_0002
  Company: barclays
  Product: ready_made_investment
  Risk Profile: defensive
  Is Table: False
  Content: Key Investor Information

BARCLAYS

This document provides you with key investor information about thi

Batches: 100%|██████████| 1/1 [00:00<00:00, 40.70it/s]


Result 1 (score: 0.6641)
  Chunk ID: 6b1b501d2c53_table_1
  Company: barclays
  Product: ready_made_investment
  Risk Profile: cautious
  Is Table: True
  Content: It can help you to assess how the fund has been managed in the past. ## Discrete 12 month performance (%)

|   | 28.02.2025
28.02.2026 | 29.02.2024
28.02.2025 | 28.02.2023
29.02.2024 | 28.02.2022
28.0...

Result 2 (score: 0.6617)
  Chunk ID: 62f0f46d3bfb_table_1
  Company: barclays
  Product: ready_made_investment
  Risk Profile: adventurous
  Is Table: True
  Content: It can help you to assess how the fund has been managed in the past. ## Discrete 12 month performance (%)

|   | 28.02.2025
28.02.2026 | 29.02.2024
28.02.2025 | 28.02.2023
29.02.2024 | 28.02.2022
28.0...

Result 3 (score: 0.6606)
  Chunk ID: b2bd6b0affda_table_1
  Company: barclays
  Product: ready_made_investment
  Risk Profile: growth
  Is Table: True
  Content: It can help you to assess how the fund has been managed in the past. ## Discrete 12 month perfor

## Experiment 2: Metadata Filtering

Let's test filtering by company or risk profile.

In [5]:
# Filter by company
query = "What are the ongoing charges?"
query_embedding = embedder.generate_embeddings([query])[0]

print("Search without filter:")
results_no_filter = indexer.search(query_embedding, limit=3)
for r in results_no_filter:
    print(f"  {r['payload']['company']}: {r['score']:.4f}")

print("\nSearch with Barclays filter:")
results_barclays = indexer.search(query_embedding, limit=3, filters={"company": "barclays"})
for r in results_barclays:
    print(f"  {r['payload']['company']}: {r['score']:.4f}")

print("\nSearch with Scottish Widows filter:")
results_sw = indexer.search(query_embedding, limit=3, filters={"company": "scottish_widows"})
for r in results_sw:
    print(f"  {r['payload']['company']}: {r['score']:.4f}")

2026-04-25 23:47:51,347 - INFO - Generating embeddings for 1 texts...
Batches: 100%|██████████| 1/1 [00:01<00:00,  1.04s/it]

Search without filter:
  scottish_widows: 0.4607
  lloyds: 0.4607
  barclays: 0.4041

Search with Barclays filter:
  barclays: 0.4041
  barclays: 0.4041
  barclays: 0.4041

Search with Scottish Widows filter:
  scottish_widows: 0.4607
  scottish_widows: 0.3937
  scottish_widows: 0.3450


## How Persistence Works

When you run this notebook, the embeddings are NOT regenerated:

1. **Embeddings are stored** in `data/qdrant_storage/` on disk
2. **Qdrant loads them** when you connect with `persistence_path`
3. **No need to regenerate** - just query the existing index!

**Benefits**:
- Fast startup (no need to regenerate 474 embeddings)
- Consistent results (same embeddings every time)
- Can share the index with other notebooks/scripts

## Experiment 3: [Add Your Experiment Here]

Use this cell to implement your own experiments:
- Hybrid retrieval (dense + sparse)
- Reranking
- Query expansion
- Different chunk sizes
- Etc.

In [ ]:
# Your experiment code here
pass

## Results Logging

Log your experiment results here for comparison.

In [ ]:
# Example: Log results
experiment_log = {
    "experiment_id": "exp_001",
    "description": "Baseline: Dense retrieval only",
    "date": "2026-04-25",
    "results": {
        "query_1_score": 0.85,
        "query_2_score": 0.72,
        # Add more results
    }
}

# Save to file
log_path = Path("../data/evaluation_results/experiments.json")
log_path.parent.mkdir(parents=True, exist_ok=True)

existing_logs = []
if log_path.exists():
    with open(log_path, "r") as f:
        existing_logs = json.load(f)

existing_logs.append(experiment_log)

with open(log_path, "w") as f:
    json.dump(existing_logs, f, indent=2)

print(f"Logged experiment results to {log_path}")